# DVDT401. Codebase Overview

This tutorial provides developers orientation on how the different parts of the DeepTrack2 codebase work together, with a focus on the lower-level modules and core functionalities.

This tutorial should ideally be read while checking the [deeptrack directory structure](https://github.com/DeepTrackAI/DeepTrack2/tree/develop/deeptrack) in order to provide perspective.

The following topics will be covered:

- Module structure

- Low-level modules, such as `core` and `image`.

- The `features` module.

- The `image` module.

- High-level modules, such as `holography` and `optics`.

- Utility and handle modules.

## 1. What is a Module?

A module in the DeepTrack2 framework is a `.py` source code file 
containing classes and methods that implement functionalities in a _modular_ way, making maintenance and debugging easier. Inheritance is used heavily in the design of DeepTrack2, and understanding the module hierarchy is crucial.

## 2. Module Structure

Each module should be clearly documented in the beginning.

Often modules contain some **abstract classes** to provide a standardized implementation of core functionalities (for example, image transformation) to ensure consistent output formatting across all derived classes in the module (for example, to ensure that the format of the output image is the same).

A typical module is structured like the following:

```python

"""Example module.

...

"""

from deeptrack.backend import BackendClass


class AbstractClass(BackendClass):
    """Define abstract class that inherits from some backbone module.

    """

    def __init__(...):
        super().__init__(...)
    
    def process():
        self.get()


class AdditionCase(AbstractClass):
    """Define concrete implementation of the abstract class.

    """
    
    def __init__(...):
        super().__init__(...)
    
    # Define abstract method.
    def get(...):
        return 1 + 1

```

## 3. Type Hints for Code Readability

As type hints are used extensively to improve code readability in accordance with the [style guide](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/tutorials/4-developers/DTDV411_style.ipynb), DeepTrack2 introduces a few custom type hints for internal use, which are declared in the [types.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/types.py) module.

Currently there are three type aliases:

- `PropertyLike`: Alias representing either a value of generic type `T` or a callable function returning a value of generic type `T`.

- `ArrayLike`: Alias for array-like structures (e.g., tuples, lists, numpy arrays, torch tensors).

- `NumberLike`: Alias for numeric types, including scalars and arrays (e.g., numpy 
    arrays, torch tensors).

You can incorporate these type hints like the following:

```python
from __future__ import annotations

from .types import ArrayLike, PropertyLike

def ClassName():

    def get(
            self : ClassName,
            image: ArrayLike,
            uses: PropertyLike[int],
            storage: PropertyLike[int],
            **kwargs
    ) -> List[ArrayLike]:

```

## 4. Low-Level Modules — backend and sources

This section covers the low-level modules in DeepTrack2 found in the [backend](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/backend/) and [sources](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/sources/) folders.


### 4.1. Backend is the Backbone of DeepTrack2

The [backend](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/backend/) directory contains the code that forms the backbone of the DeepTrack2 framework.

It consists of six modules:

- [_config.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/backend/_config.py) provides the funcionalities to manage the computational backend (which can be switched between NumPy and PyTorch), the computational device (CPU, GPU, MPS, etc.), and the image wrapper.

    The [array_api_compat_ext](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/backend/array_api_compat_ext/) directory contains the files necessary to manage the computational backends.

- [core.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/backend/core.py) provides the core DeepTrack2 classes to manage and process data on a fundamental level.  

    In particular, the [core.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/backend/core.py) module contains the `DeepTrackNode` class which is used to represent a node in a computation graph, which when used together with the `DeepTrackDataObject` class can store data and compute new data based on its dependencies and child nodes. These classes track dependencies and validate data with ID and index addresses.  

    The [core.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/backend/core.py) module also provides the base class for the [features.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/features.py) module, which is the largest module in DeepTrack2 in terms of code volume, and provides the base class for all other modules in the deeptrack directory; the only exceptions are [image.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/image.py) and [properties.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/properties.py).

- [mie.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/backend/mie.py) provides functions to perform Mie scattering calculations often used in simulations.

    In particular, it provides methods to compute coefficients for both spherical and stratified spherical harmonics, and to calculate the spherical harmonics of the Mie field with an iterative method.

- [pint_definition.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/backend/pint_definition.py)extends Pint's default definitions by introducing project-specific constants and unit modifications for flexible calculations in the context of DeepTrack2.

- [polynomials.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/backend/polynomials.py) provides a set of functions which compute Bessel and Riccati-Bessel polynomials and their derivatives.

- [units.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/backend/units.py) provides unit management and conversions for simulations involving voxel dimensions and scaling factors.

### 4.2. Managing data sources

The [sources](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/sources/) directory contains the code used to provide utility classes to manage image data sources and random number generators.

It contains three modules:

- [base.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/sources/base.py) extends `DeepTrackNode` objects to represent sources of data, and enables data validity checking.

- [folder.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/sources/folder.py) introduces utilities to organize sources in directories with labeling and source splitting.

- [rng.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/sources/rng.py) extends both the standard library rng and NumPy rng to let the user instance as many generators as desired with unique seeds.

## 5. Data Structures and Transformations

The [features.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/features.py) module introduces the `Feature` and `StructuralFeature` base classes, which form the base for all features and their implementations in DeepTrack2.

A `Feature` is a building block of a data processing pipeline, representing a transformation applied to data.
Often features are used to transform images. Some examples of these image transformations are: rotations or deformations; noise addition or background illumination; non-additive elements, such as Poisson noise.
For example, in the [augmentations.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/augmentations.py) module: an augmentation that rotates an image is implemented as a subclass to `Feature`.

The `StructuralFeature` class further extends `Feature` by adding logical structures, such as branches or chaining. It enables the construction of data pipelines with more advanced requirements.

## 6. Adding Attributes and Properties to Features

The [properties.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/properties.py) module introduces the `Property`, `SequentialProperty`, and `PropertyDict` base classes, which, when used in combination with the [features.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/features.py) module, let the user add values to a `Feature`.

The value of a `Property` has no datatype restrictions and can represent a constant, a function, a list, an `Image`, etc.

`PropertyDict` represents a dictionary with `Property` elements.

`SequentialProperty` extends the `Property` class to enables sequential updates to handle scenarios where the property’s value evolves over discrete steps, such as frames in a video or datapoints in a time series.

## 7. Containers for Array-Like Structures

The [image.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/image.py) module only contains a single class, namely `Image`. This class serves as a wrapper for array-like data and provides a unified interface for array operations and property management with NumPy and PyTorch compatibility.

Several utility functions are also included, which can be used to manipulate `Image` objects within pipelines, such as image coercion to ensure a consistent type across a series of images, or padding to optimize Fast Fourier Transform performance.


## 8. High-Level Modules

The remaoning modules jointly implement the main functionality of DeepTrack2, which is synthetic data generation using simulations.

All the classes in the following modules extend `Feature` and utilize `Image` objects as containers, as all of these represent transformations in some way:

- [scatterers.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/scatterers.py) provides a framework for implementing light-scattering objects.

- [optics.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/optics.py) provides a framework for simulating the optics of a variety of microscopy setups such as darkfield, fluorescence, or brightfield, as well as for incorporating the interaction between a sample (often a scatterer object) and a microscopy setup.

- [holography.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/holography.py) provides features for managing optical fields with Fourier transforms and propagation matrices for usage in simulation pipelines and holographic reconstructions.

- [aberrations.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/aberrations.py) provides a framework to simulate optical aberrations in microscopy setups by implementing Zernike polynomials and Gaussian pupil apodization.

- [augmentations.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/augmentations.py) provides a framework to augment data with various transformations and image manipulation techniques such as rotations, deformations, cropping, padding.

- [noises.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/noises.py) provides a framework to add various types of additive noise to images as well as Poisson noise based on signal-to-noise ratio.


## 9. Handle and Utility Modules

The following modules play a mediating role between external libraries and DeepTrack2, such as integrating Pytorch classes or Numpy functions, as well as utilities for type consistency and radial center calculations.

### 9.1. Numpy
The following modules provide handles to enable the use of NumPy functions with DeepTrack2 objects as well as mathematical utilities:

- [elementwise.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/elementwise.py) provides handles to perform various elementary NumPy functions elementwise and sequentially when using `Feature` objects. For example, some of these functions include trigonometric, hyperbolic, rounding, exponents.

- [math.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/math.py) provides utilities for various types of normalization, blurring, pooling, and resizing of `Image` objects.

- [statistics.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/statistics.py) provides handles to perform various statistical NumPy functions on a given `Feature` objects along a given axis. 

### 9.2. Pytorch
Located in the [pytorch](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/pytorch/) directory, there are two modules to facilitate PyTorch integration with DeepTrack2 objects:

- [pytorch.data.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/pytorch/data.py) extends the PyTorch [Dataset](https://pytorch.org/tutorials/beginner/basics/data_tutorial.html) class to work with DeepTrack2 `Image` objects.

- [pytorch.features.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/pytorch/features.py) extends `Feature` to be able to convert an input to a PyTorch [Tensor](https://pytorch.org/docs/stable/tensors.html).

### 9.3. Deeplay

The init file in [deeplay/\_\_init__.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/deeplay/__init__.py) enables users to import Deeplay from DeepTrack2 with:

```python 
import deeptrack.deeplay as dl
```

### 9.4. Other Modules

- [utils.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/utils.py) provides various utilities to streamline common operations, ensuring type and argument consistency with various check methods and safe call.

- [extras.radialcenter.py](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/extras/radialcenter.py)introduces a single function to calculate the center location of an intensity distribution with a least-squares method.

## 10. Unit testing

The unit testing scripts are found in the [tests](https://github.com/DeepTrackAI/DeepTrack2/blob/develop/deeptrack/tests/) directory, and follow the same structure as the [deeptrack](https://github.com/DeepTrackAI/DeepTrack2/tree/develop/deeptrack) directory with a test script for each module.

You can run all unit tests from the Python command line typing the following command in the root folder of the DeepTrack2 repository.

```bash
python -m unittest discover -v deeptrack.tests
```